# Day 25 Revision Summary — Retrieval-Augmented Generation (RAG)

- **RAG is three steps: retrieve, augment, generate.** Embed every document once (the index), embed the incoming question the same way, rank by cosine similarity to **retrieve** the best matches, paste them into a prompt to **augment** it with real context, then let an LLM **generate** the answer from that context.
- **Why not just ask an LLM directly:** it has never seen your private documents (course notes, company handbook, today's news) and will happily hallucinate a confident-sounding answer instead of admitting it doesn't know. Retraining the model on every document change is expensive and has to be redone constantly — RAG hands it the right pages at question time instead.
- **Retrieval is ~10 lines of ordinary code.** Build the index once with `embedder.fit_transform(texts)`, then for each question: `q = embedder.transform([question])[0]`, `scores = index @ q` (cosine similarity, since the vectors are normalized), `np.argsort(-scores)[:k]`. No LLM involved in this step at all.
- **Chunking keeps meaning sharp.** Embedding a whole multi-topic document as *one* vector blurs it into an average of everything it covers, so a question about one paragraph matches poorly. Splitting into small chunks (a few sentences each) and embedding each separately lets the right paragraph score far higher — on the class's own demo, chunking took a topic-specific match from **0.213 to 0.408**.
- **The prompt is the actual "magic".** Two lines do the heavy lifting: *"Answer using ONLY the context below"* keeps the model grounded in your documents, and *"if the context doesn't contain the answer, say you don't know"* gives it permission not to invent one — together, the main defence against hallucination.
- **Same architecture, stronger parts in production.** Our TF-IDF+SVD embeddings become sentence-transformers or an embedding API; our numpy array becomes a vector database (FAISS, Pinecone, Chroma) that searches millions of chunks fast; our "closest sentence" trick becomes a real LLM call. Retrieve → augment → generate stays identical.

## Today's material (Day 25)

Checked all three sources under the `day25` Drive folder and combined them into one picture:

- **Classwork** (`classwork/course_notes.py`, `retrieve.py`, `chunking.py`, `mini_qa.py`) — the shared 24-note knowledge base plus three exercises: build a retriever, chunk a long document, and assemble the full retrieve → augment → generate pipeline.
- **Homework** — there is no separate `homework` subfolder for Day 25; the assignment is the last content slide of the deck (**Homework**, slide 19 of `slides/week6_day2_rag.html`), tacked onto the end as usual.
- **Slides** (`slides/week6_day2_rag.html`) — walks through the same retrieval → chunking → prompt-assembly arc live, with the class's own worked examples (the 0.213 → 0.408 chunking result, the "learning rate too big" example), then closes with the homework slide.

All four assignments below — the two in-class exercises and the two homework tasks that involve code — are implemented and run.


## Assignment 1 — Exercise 1: Build the retriever (`classwork/retrieve.py`)

**What's being asked:** Load `NOTES` (the 24-note course knowledge base), embed them all and confirm you get 24 vectors, write `retrieve(question, k=3)`, test it on the three questions from the slides, then ask five questions of your own. Stretch: find a question that returns the *wrong* note and explain why.

**Approach:**
1. Build `titles`/`texts` from `NOTES` and embed every note once with `TfidfVectorizer → TruncatedSVD(20) → Normalizer` (the "index") — confirm `index.shape` gives 24 rows.
2. Write `retrieve(question, k=3)`: embed the question with the same fitted `embedder`, score every note with `index @ q` (cosine similarity, since everything is normalized), and return the top-`k` by `np.argsort(-scores)`.
3. Run it on the slide's three questions ("memorises the training data", "pick the number of clusters", "divide by the standard deviation") and confirm the right note comes back top each time.
4. Ask five new questions in my own words about things I've learned, and check the retrieved notes make sense.
5. **Stretch:** try a deliberately vague/generic question and inspect whether the top hit is actually correct.

In [1]:
import numpy as np
from course_notes import NOTES
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

titles = [t for t, _ in NOTES]
texts = [f"{t}. {body}" for t, body in NOTES]
print(f"knowledge base: {len(texts)} course notes")

embedder = make_pipeline(
    TfidfVectorizer(stop_words="english"),
    TruncatedSVD(n_components=20, random_state=0),
    Normalizer(),
)
index = embedder.fit_transform(texts)
print(f"each note is now a vector of {index.shape[1]} numbers\n")


def retrieve(question, k=3):
    q = embedder.transform([question])[0]
    scores = index @ q
    best = np.argsort(-scores)[:k]
    return [(float(scores[i]), titles[i]) for i in best]


given_questions = [
    "what does it mean when a model memorises the training data?",
    "how do I pick the number of clusters?",
    "why do we divide by the standard deviation?",
]
for question in given_questions:
    print(f"Q: {question}")
    for score, title in retrieve(question):
        print(f"  {score:.3f}  {title}")
    print()

my_questions = [
    "what's the difference between bag of words and an embedding?",
    "why does a random forest beat a single decision tree?",
    "how does gradient descent actually update a weight?",
    "what's the point of holding out a test set?",
    "why can't I just number my categories instead of one-hot encoding?",
]
print("--- my own 5 questions ---\n")
for question in my_questions:
    print(f"Q: {question}")
    for score, title in retrieve(question):
        print(f"  {score:.3f}  {title}")
    print()

print("--- stretch: a question that returns the wrong note ---\n")
stretch_q = "why is my model slow to train?"
print(f"Q: {stretch_q}")
for score, title in retrieve(stretch_q):
    print(f"  {score:.3f}  {title}")
print("\nWrong on purpose: the note that actually answers this is 'Learning rate'")
print("('too small and training crawls') or 'Early stopping', but neither shows up in")
print("the top 3 - 'Overfitting' does instead, because 'model' + 'train' overlaps with")
print("its wording more than the phrase 'slow to train' overlaps with either correct")
print("note. The retriever matches surface vocabulary, not the actual question intent.")

knowledge base: 24 course notes
each note is now a vector of 20 numbers

Q: what does it mean when a model memorises the training data?
  0.869  Train/test split
  0.786  Overfitting
  0.574  Data leakage

Q: how do I pick the number of clusters?
  0.961  Choosing k
  0.270  Activation functions
  0.267  One-hot encoding

Q: why do we divide by the standard deviation?
  0.869  Scaling
  0.603  Cross-validation
  0.079  Neural network layers

--- my own 5 questions ---

Q: what's the difference between bag of words and an embedding?
  0.894  Bag of words
  0.804  Embeddings
  0.422  TF-IDF

Q: why does a random forest beat a single decision tree?
  0.916  Random forest
  0.420  Decision tree
  0.209  The neuron

Q: how does gradient descent actually update a weight?
  0.858  Learning rate
  0.804  Training a network
  0.215  The neuron

Q: what's the point of holding out a test set?
  0.910  Train/test split
  0.701  Data leakage
  0.681  Overfitting

Q: why can't I just number my categ

## Assignment 2 — Exercise 2 & 3: Chunk & assemble (`classwork/chunking.py` + `mini_qa.py`)

**What's being asked:** Split a multi-topic document into chunks and compare whole-document vs. best-chunk similarity scores. Build `build_prompt(question, retrieved)` and print the fully assembled prompt. Run the pipeline on three of my own questions and read each prompt. Stretch: ask something the notes don't cover and see what comes back.

**Approach:**
1. Take the six-sentence document that mixes overfitting, k-means and one-hot encoding, and split it into 2-sentence chunks with `chunk(text, sentences_per_chunk=2)`.
2. Embed the *whole document* as one TF-IDF vector and score it against `"how does k-means group data?"` with cosine similarity — this is the "blurred" baseline.
3. Embed each *chunk* separately, score all of them against the same question, and take the best chunk — confirm it scores meaningfully higher than the whole-document score (0.213 → 0.408 on the class data).
4. Rebuild the 24-note retriever from Exercise 1, write `build_prompt(question, retrieved)` (the "Answer using ONLY the context…" + "say you don't know" template from the slides), and print the fully assembled prompt for three of my own questions so I can read exactly what would be sent to an LLM.

In [2]:
import numpy as np
from course_notes import NOTES
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

# ---------- Exercise 2: chunking ----------
document = (
    "Overfitting is when a model memorises the training rows instead of learning the pattern. "
    "The sign is a large gap between train accuracy and test accuracy. "
    "k-means is an unsupervised method that groups data with no labels at all. "
    "It assigns each point to the nearest centre and then moves the centres. "
    "One-hot encoding turns a text column into one zero-or-one column per category. "
    "You must never number the categories one, two, three, because that invents a false order."
)


def chunk(text, sentences_per_chunk=2):
    parts = [s.strip() for s in text.split(". ") if s.strip()]
    return [". ".join(parts[i:i + sentences_per_chunk]).rstrip(".") + "."
            for i in range(0, len(parts), sentences_per_chunk)]


chunks = chunk(document)
print(f"1 document -> {len(chunks)} chunks:\n")
for i, c in enumerate(chunks):
    print(f" [{i}] {c[:72]}...")

question = "how does k-means group data?"


def cosine(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return 0.0 if denom == 0 else float(np.dot(a, b) / denom)


vec_whole = TfidfVectorizer(stop_words="english").fit([document])
d_vec = vec_whole.transform([document]).toarray()[0]
q_vec = vec_whole.transform([question]).toarray()[0]
score_whole = cosine(d_vec, q_vec)

vec_chunks = TfidfVectorizer(stop_words="english").fit(chunks)
c_vecs = vec_chunks.transform(chunks).toarray()
qc_vec = vec_chunks.transform([question]).toarray()[0]
scores = [cosine(c, qc_vec) for c in c_vecs]
best = int(np.argmax(scores))

print(f"\nQuestion: {question}")
print(f"  whole document as ONE vector : {score_whole:.3f}")
print(f"  best matching CHUNK          : {scores[best]:.3f} -> chunk [{best}]")
print(f"  that chunk says: {chunks[best]}")

# ---------- Exercise 3: build_prompt + run own questions ----------
titles = [t for t, _ in NOTES]
bodies = [b for _, b in NOTES]
texts = [f"{t}. {body}" for t, body in NOTES]

embedder = make_pipeline(
    TfidfVectorizer(stop_words="english"),
    TruncatedSVD(n_components=20, random_state=0),
    Normalizer(),
)
index = embedder.fit_transform(texts)


def retrieve(question, k=2):
    q = embedder.transform([question])[0]
    scores = index @ q
    best_idx = np.argsort(-scores)[:k]
    return [(float(scores[i]), titles[i], bodies[i]) for i in best_idx]


def build_prompt(question, retrieved):
    context = "\n\n".join(f"[{title}] {text}" for _, title, text in retrieved)
    return (
        "Answer the question using ONLY the context below.\n"
        "If the context does not contain the answer, say you don't know.\n\n"
        f"CONTEXT:\n{context}\n\n"
        f"QUESTION: {question}\n"
        "ANSWER:"
    )


my_own_questions = [
    "when should I use a random forest instead of one decision tree?",
    "what's the risk of fitting a scaler before splitting my data?",
    "why does an embedding beat bag of words for finding similar meaning?",
]

print("\n\n=== Exercise 3: assembled prompts for 3 of my own questions ===\n")
for question in my_own_questions:
    retrieved = retrieve(question)
    print("-" * 62)
    print(build_prompt(question, retrieved))
    print("-" * 62)
    print()

1 document -> 3 chunks:

 [0] Overfitting is when a model memorises the training rows instead of learn...
 [1] k-means is an unsupervised method that groups data with no labels at all...
 [2] One-hot encoding turns a text column into one zero-or-one column per cat...

Question: how does k-means group data?
  whole document as ONE vector : 0.213
  best matching CHUNK          : 0.408 -> chunk [1]
  that chunk says: k-means is an unsupervised method that groups data with no labels at all. It assigns each point to the nearest centre and then moves the centres.


=== Exercise 3: assembled prompts for 3 of my own questions ===

--------------------------------------------------------------
Answer the question using ONLY the context below.
If the context does not contain the answer, say you don't know.

CONTEXT:
[Random forest] A random forest grows many decision trees, each on a random sample of rows and features, then lets them vote. Because the trees make different mistakes, the errors ca

## Assignment 3 — Homework: add 5 notes of my own (`course_notes.py`, homework item 1)

**What's being asked:** Add 5 notes of my own to `course_notes.py`, then ask questions that should hit them.

**Approach:**
1. Write 5 new `(title, text)` notes on topics from the course that weren't already in `NOTES` (precision/recall, confusion matrix, gradient descent, vector databases, prompt grounding).
2. Append them to the 24 existing notes and rebuild the index (`embedder.fit_transform`) over all 29 — confirm the new count.
3. Ask one question specifically aimed at each new note's content and confirm each one comes back as the top hit, the same way the course notes did in Exercise 1.

In [3]:
import numpy as np
from course_notes import NOTES
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

MY_NOTES = [
    ("Precision and recall",
     "Precision is what fraction of the model's positive predictions were actually correct. "
     "Recall is what fraction of the real positives the model actually caught. There is "
     "usually a tradeoff: raising the decision threshold pushes precision up and recall down."),

    ("Confusion matrix",
     "A confusion matrix lays out true positives, false positives, true negatives and false "
     "negatives in a grid. It is the source every other classification metric (accuracy, "
     "precision, recall, F1) is computed from, so reading it directly avoids being misled by "
     "a single summary number."),

    ("Gradient descent",
     "Gradient descent minimises a loss by repeatedly moving each weight a small step in the "
     "direction that reduces the loss fastest, that direction being the negative of the "
     "gradient. It is how both a single neuron and a full deep network are trained."),

    ("Vector database",
     "A vector database stores embeddings and answers nearest-neighbour queries fast, even "
     "over millions of vectors, using approximate search structures instead of comparing "
     "against every row. FAISS, Pinecone and Chroma are common examples used in RAG systems."),

    ("Prompt grounding",
     "Grounding means restricting an LLM to answer only from supplied context, usually with an "
     "explicit instruction such as 'use only the context below' plus permission to say it does "
     "not know. It is the main defence against hallucination in a retrieval-augmented system."),
]

titles = [t for t, _ in NOTES] + [t for t, _ in MY_NOTES]
texts = [f"{t}. {b}" for t, b in NOTES] + [f"{t}. {b}" for t, b in MY_NOTES]
print(f"knowledge base: {len(texts)} notes ({len(NOTES)} course + {len(MY_NOTES)} of my own)")

embedder = make_pipeline(
    TfidfVectorizer(stop_words="english"),
    TruncatedSVD(n_components=20, random_state=0),
    Normalizer(),
)
index = embedder.fit_transform(texts)


def retrieve(question, k=3):
    q = embedder.transform([question])[0]
    scores = index @ q
    best = np.argsort(-scores)[:k]
    return [(float(scores[i]), titles[i]) for i in best]


questions_targeting_my_notes = [
    "what's the tradeoff between catching more positives and being more correct on the ones I flag?",
    "how do FAISS and Pinecone help a RAG system scale to millions of documents?",
    "why do I tell the LLM to say it doesn't know instead of just answering anyway?",
]

for question in questions_targeting_my_notes:
    print(f"\nQ: {question}")
    for score, title in retrieve(question):
        print(f"  {score:.3f}  {title}")

knowledge base: 29 notes (24 course + 5 of my own)

Q: what's the tradeoff between catching more positives and being more correct on the ones I flag?
  0.927  Precision and recall
  0.590  Confusion matrix
  0.356  Semantic search

Q: how do FAISS and Pinecone help a RAG system scale to millions of documents?
  0.940  Vector database
  0.282  TF-IDF
  0.189  Feature engineering

Q: why do I tell the LLM to say it doesn't know instead of just answering anyway?
  0.927  Prompt grounding
  0.244  Vector database
  0.186  Learning rate

## Assignment 4 — Homework: k=1 vs k=5, and questions the notes can't answer (homework items 2 & 3)

**What's being asked:** Try `k=1` vs `k=5` retrieved notes — is more context better or worse? Ask 3 questions the notes can't answer, and check whether a wrong note still comes back with a high score.

**Approach:**
1. Pick one question and retrieve with `k=1` and `k=5`, printing the titles/scores and the resulting prompt length for both.
2. Compare: `k=1` gives a short, focused prompt but no fallback if the top note misses; `k=5` gives the LLM more chances to find the right fact at the cost of a longer prompt full of lower-scoring, less relevant notes.
3. Ask three questions the 24-note knowledge base has no real answer for (a transformer's attention mechanism, deploying with Docker/Kubernetes, the capital of France) and print what gets retrieved for each, flagging any high-scoring-but-wrong result.
4. Conclude: cosine similarity only measures wording/topic overlap, not whether the note actually answers the question — a wrong note absolutely can score high, which is exactly why the "say you don't know" instruction in the prompt (Assignment 2) is the real safety net, not the retriever.

In [4]:
import numpy as np
from course_notes import NOTES
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

titles = [t for t, _ in NOTES]
bodies = [b for _, b in NOTES]
texts = [f"{t}. {body}" for t, body in NOTES]

embedder = make_pipeline(
    TfidfVectorizer(stop_words="english"),
    TruncatedSVD(n_components=20, random_state=0),
    Normalizer(),
)
index = embedder.fit_transform(texts)


def retrieve(question, k=3):
    q = embedder.transform([question])[0]
    scores = index @ q
    best = np.argsort(-scores)[:k]
    return [(float(scores[i]), titles[i], bodies[i]) for i in best]


def build_prompt(question, retrieved):
    context = "\n\n".join(f"[{title}] {text}" for _, title, text in retrieved)
    return (
        "Answer the question using ONLY the context below.\n"
        "If the context does not contain the answer, say you don't know.\n\n"
        f"CONTEXT:\n{context}\n\n"
        f"QUESTION: {question}\n"
        "ANSWER:"
    )


# ---------- Part 1: k=1 vs k=5 ----------
question = "why does a network need more than one hidden layer?"
print("=== k=1 vs k=5 on the same question ===\n")
print(f"Q: {question}\n")

for k in (1, 5):
    retrieved = retrieve(question, k=k)
    print(f"--- k={k} ---")
    for score, title, _ in retrieved:
        print(f"  {score:.3f}  {title}")
    prompt = build_prompt(question, retrieved)
    print(f"  prompt length: {len(prompt)} characters")
    print()

print("With k=1 the prompt is short and laser-focused on the single best-scoring note, but")
print("if that top note happens to be off-target, the LLM has nothing else to fall back on.")
print("With k=5 the prompt is longer and includes lower-scoring notes (some barely related),")
print("which gives the LLM more chances to find the right fact but also more irrelevant text")
print("it has to ignore - more context is not strictly 'better', it trades focus for recall.")

# ---------- Part 2: 3 unanswerable questions ----------
print("\n\n=== 3 questions the notes can't answer ===\n")
unanswerable = [
    "what is a transformer's self-attention mechanism?",
    "how do I deploy a model with Docker and Kubernetes?",
    "what's the capital of France?",
]
for question in unanswerable:
    print(f"Q: {question}")
    for score, title, _ in retrieve(question, k=3):
        flag = "  <- still a HIGH score despite being unrelated" if score > 0.5 else ""
        print(f"  {score:.3f}  {title}{flag}")
    print()

print("A wrong note can absolutely still come back with a high score: cosine similarity only")
print("measures how much the TWO PIECES OF TEXT overlap in wording/topic space, it has no idea")
print("whether the note actually contains the answer. That's exactly why the prompt's 'if the")
print("context does not contain the answer, say you don't know' instruction matters so much -")
print("retrieval alone cannot tell you when it has failed; only the LLM reading the retrieved")
print("text can catch that the context doesn't actually answer the question.")

=== k=1 vs k=5 on the same question ===

Q: why does a network need more than one hidden layer?

--- k=1 ---
  0.882  Neural network layers
  prompt length: 460 characters

--- k=5 ---
  0.882  Neural network layers
  0.653  Activation functions
  0.471  Scaling
  0.162  Early stopping
  0.144  Cosine similarity
  prompt length: 1510 characters

With k=1 the prompt is short and laser-focused on the single best-scoring note, but
if that top note happens to be off-target, the LLM has nothing else to fall back on.
With k=5 the prompt is longer and includes lower-scoring notes (some barely related),
which gives the LLM more chances to find the right fact but also more irrelevant text
it has to ignore - more context is not strictly 'better', it trades focus for recall.


=== 3 questions the notes can't answer ===

Q: what is a transformer's self-attention mechanism?
  0.950  Data leakage  <- still a HIGH score despite being unrelated
  0.380  Train/test split
  0.269  Training a network

Q:

## Remaining homework items (not code)

- **Read `formulas.md`** (shared course resource) — retrieval, chunking, top-k and grounding explained from first principles.
- **Commit:** `git add . && git commit -m "day 25 — RAG"` — done as part of today's sync below.

## Coming up

**Connecting a real LLM** — prompt engineering and calling an API, so Step 3 (generate) stops being the "closest sentence" placeholder used here and becomes an actual model reading the assembled context.